In [ ]:
# ==========================================
# AIS Data Cleaning & Frequency Analysis
# ==========================================
# This script performs two key operations on the raw AIS dataset:
# 1. Quality Filtering: Removes unreliable or stationary data points based on:
#    - SOG > 0.5 (Removes docked/anchored vessels)
#    - Draft > 0 (Ensures valid vessel configuration)
#    - Valid Coordinates (Removes default/error values like 91, 181)
#    - Valid Heading (Removes default 511 values)
#
# 2. Frequency Analysis: Calculates how many rows remain at different
#    "minimum appearance" cutoffs (e.g., >= 100, >= 1000 rows per vessel)
#    to help determine the optimal threshold for reducing file size.
# ==========================================



import pandas as pd
import numpy as np

# --- Settings ---
input_file = 'AIS_2024_01_01.csv'

print(f"Loading {input_file}...")
# Loading specific columns saves memory, but for safety we load all to check column names
df = pd.read_csv(input_file, low_memory=False)

original_count = len(df)
print(f"Original Row Count: {original_count:,}")

# --- 1. Apply Quality Filters ---
print("\nApplying Quality Filters...")

# Filter 1: Remove Invalid Coordinates (91 and 181 are AIS error codes)
# We use absolute value to catch -91 or -181 just in case
valid_geo = (df['LAT'].abs() <= 90) & (df['LON'].abs() <= 180) & \
            (df['LAT'] != 91) & (df['LON'] != 181)

# Filter 2: Remove Stationary / GPS Jitter (SOG > 0.5 knots)
# This removes docked and anchored ships
moving_ships = df['SOG'] > 0.5

# Filter 3: Valid Draft (Draft > 0)
# Removes vessels that haven't configured their transponder details
valid_draft = df['Draft'] > 0

# Filter 4 (Optional): Remove Invalid Heading (511 is 'Not Available')
valid_heading = df['Heading'] != 511

# COMBINE ALL FILTERS
# We use bitwise AND (&) to satisfy ALL conditions
quality_mask = valid_geo & moving_ships & valid_draft & valid_heading

df_clean = df[quality_mask]

# --- 2. Report on Quality Filtering ---
clean_count = len(df_clean)
removed_count = original_count - clean_count
print(f"Rows removed by quality filters: {removed_count:,} ({(removed_count/original_count)*100:.1f}%)")
print(f"Rows remaining (High Quality):   {clean_count:,}")

if clean_count == 0:
    print("\n[CRITICAL] All rows were filtered out! Check your column names (case sensitive).")
    print(f"Columns found: {df.columns.tolist()}")
    exit()

# --- 3. Generate New Cutoff Statistics ---
print("\n--- Re-evaluating Cutoffs on Cleaned Data ---")

# Count vessel appearances in the CLEAN dataset
vessel_counts = df_clean['MMSI'].value_counts()
total_vessels = len(vessel_counts)

print(f"Total Unique Moving Vessels: {total_vessels:,}")

cutoffs = [1, 5, 10, 50, 100, 200, 500, 1000]

print(f"\n{'Cutoff (>= X)':<15} | {'Rows Kept':<15} | {'% Data Kept':<15} | {'Vessels Kept':<15}")
print("-" * 65)

for cut in cutoffs:
    # Filter logic: Keep if count is Greater than or Equal to 'cut'
    kept_vessels = vessel_counts[vessel_counts >= cut]
    
    # Calculate stats based on the ALREADY CLEANED data
    rows_kept = kept_vessels.sum()
    percent_kept = (rows_kept / clean_count) * 100
    num_vessels_kept = len(kept_vessels)
    
    print(f"{cut:<15} | {rows_kept:<15,} | {percent_kept:.2f}%          | {num_vessels_kept:<15,}")

Loading AIS_2024_01_01.csv...
Original Row Count: 7,296,275

Applying Quality Filters...
Rows removed by quality filters: 6,332,156 (86.8%)
Rows remaining (High Quality):   964,119

--- Re-evaluating Cutoffs on Cleaned Data ---
Total Unique Moving Vessels: 2,771

Cutoff (>= X)   | Rows Kept       | % Data Kept     | Vessels Kept   
-----------------------------------------------------------------
1               | 964,119         | 100.00%          | 2,771          
5               | 963,636         | 99.95%          | 2,539          
10              | 963,043         | 99.89%          | 2,452          
50              | 953,991         | 98.95%          | 2,116          
100             | 936,120         | 97.10%          | 1,870          
200             | 879,429         | 91.22%          | 1,481          
500             | 630,742         | 65.42%          | 745            
1000            | 254,680         | 26.42%          | 221            


In [ ]:
import pandas as pd

# ==========================================
# AIS Data Cleaning & Reduction Script
# ==========================================
# This script processes the raw AIS dataset to create a smaller, high-quality file.
#
# Process Flow:
# 1. Quality Filtering: Removes unreliable or stationary data points.
#    - SOG > 0.5 (Removes docked/anchored vessels)
#    - Draft > 0 (Ensures valid vessel configuration)
#    - Valid Coordinates (Removes default/error values like 91, 181)
#    - Valid Heading (Removes default 511 values)
#
# 2. Frequency Filtering:
#    - Calculates the number of valid rows per vessel (MMSI).
#    - Removes any vessel that appears fewer than 1000 times.
#    - Result: A dataset of only "active" vessels with rich history.
# ==========================================
# 3. Multiple run:
#    - Run this code multiple time by changing the AIS data files from day 01 to day 05 of January 2024
#      AIS_2024_01_01.csv, AIS_2024_01_02.csv, AIS_2024_01_03.csv, AIS_2024_01_04.csv, AIS_2024_01_05.csv
# ==========================================


# --- Settings ---
input_file = 'AIS_2024_01_01.csv'
output_file = 'AIS_2024_01_01_cleaned_HighFreq.csv'
frequency_cutoff = 1000  # Keep vessels with >= 1000 rows

print(f"Loading {input_file}...")
df = pd.read_csv(input_file, low_memory=False)

original_rows = len(df)
print(f"Original Row Count: {original_rows:,}")

# --- Step 1: Quality Filtering ---
print("\n--- Step 1: Applying Quality Filters ---")

# Define masks for valid data
valid_geo = (df['LAT'].abs() <= 90) & (df['LON'].abs() <= 180) & \
            (df['LAT'] != 91) & (df['LON'] != 181)
moving_ships = df['SOG'] > 0.5
valid_draft = df['Draft'] > 0
valid_heading = df['Heading'] != 511

# Apply all filters at once
df_quality = df[valid_geo & moving_ships & valid_draft & valid_heading].copy()

# Report stats after Step 1
quality_rows = len(df_quality)
rows_dropped_quality = original_rows - quality_rows
print(f"Rows removed by quality filters: {rows_dropped_quality:,}")
print(f"Rows remaining (High Quality):   {quality_rows:,}")


# --- Step 2: Frequency Filtering ---
print(f"\n--- Step 2: Applying Cutoff (>= {frequency_cutoff} rows) ---")

# Calculate counts on the ALREADY cleaned data
# We use .transform to create a boolean mask directly
mask_freq = df_quality.groupby('MMSI')['MMSI'].transform('count') >= frequency_cutoff
df_final = df_quality[mask_freq]

# Report stats after Step 2
final_rows = len(df_final)
rows_dropped_freq = quality_rows - final_rows
unique_vessels = df_final['MMSI'].nunique()

print(f"Rows removed by frequency cutoff: {rows_dropped_freq:,}")
print(f"Final Row Count:                  {final_rows:,}")
print(f"Unique Vessels Remaining:         {unique_vessels:,}")


# --- Step 3: Saving ---
print(f"\nSaving final dataset to {output_file}...")
df_final.to_csv(output_file, index=False)
print("Done! Process complete.")

Loading AIS_2024_01_01.csv...
Original Row Count: 7,296,275

--- Step 1: Applying Quality Filters ---
Rows removed by quality filters: 6,332,156
Rows remaining (High Quality):   964,119

--- Step 2: Applying Cutoff (>= 1000 rows) ---
Rows removed by frequency cutoff: 709,439
Final Row Count:                  254,680
Unique Vessels Remaining:         221

Saving final dataset to AIS_2024_01_01_cleaned_HighFreq.csv...
Done! Process complete.


In [ ]:
# ==========================================
# Combining the generated files
# ==========================================
# This script combines all the AIS csv files generated from the last script, for day 01 to day 05 of January and
# create a single dataset, to be pushed to the PostgreSQL 

import pandas as pd

# 1. Define the list of files to be combined
files = [
    'AIS_2024_01_01_cleaned_HighFreq.csv',
    'AIS_2024_01_02_cleaned_HighFreq.csv',
    'AIS_2024_01_03_cleaned_HighFreq.csv',
    'AIS_2024_01_04_cleaned_HighFreq.csv',
    'AIS_2024_01_05_cleaned_HighFreq.csv'
]

# 2. Load and combine the files
# pd.read_csv(f) reads each file
# pd.concat joins them into one large table
# ignore_index=True ensures the row numbers are continuous (0, 1, 2...)
combined_df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

# 3. Save the result to a new CSV file
# index=False prevents pandas from adding an extra 'index' column to your file
output_file = 'AIS_2024_01_Combined_clean_HighFreq.csv'
combined_df.to_csv(output_file, index=False)

print(f"Process complete! {len(files)} files merged into '{output_file}'.")

Process complete! 5 files merged into 'AIS_2024_01_Combined_clean_HighFreq.csv'.
